# ViralBERT - fine-tune BERT for viral content quality (run on Kaggle GPU)

Component 1 done properly: fine-tune `bert-base-uncased` on the binary `is_viral` label,
512 tokens, 5-fold out-of-fold PR-AUC so it is directly comparable to the TF-IDF baseline (0.604).

**Setup on Kaggle:**
1. Upload `video_features.parquet` as a Kaggle Dataset.
2. Notebook Settings -> Accelerator -> **GPU T4 x2** (or any GPU).
3. Edit `DATA` below to your dataset path, then Run All.
4. Download `/kaggle/working/viralbert/` (model) + `viralbert_oof_scores.parquet` from the Output tab.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# EDIT this path to your uploaded Kaggle dataset
DATA = "/kaggle/input/viralbert-data/video_features.parquet"

df = pd.read_parquet(DATA)
y = (df["virality_score"] >= df["virality_score"].quantile(0.75)).astype(int).values
texts = df["text_all"].fillna("").tolist()
print(len(texts), "docs | viral rate:", round(float(y.mean()), 3))

In [ ]:
MODEL_NAME, MAXLEN, EPOCHS, BS, LR = "bert-base-uncased", 512, 4, 16, 2e-5
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_loader(texts_list, labels, shuffle):
    enc = tok(texts_list, truncation=True, max_length=MAXLEN, padding="max_length", return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels))
    return DataLoader(ds, batch_size=BS, shuffle=shuffle)

def train_one(tr_idx, te_idx):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    ytr = y[tr_idx]
    w = torch.tensor([1.0, float((ytr == 0).sum() / max((ytr == 1).sum(), 1))]).to(device)
    lossf = torch.nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    dl = make_loader([texts[i] for i in tr_idx], ytr, True)
    model.train()
    for ep in range(EPOCHS):
        for ids, am, lb in dl:
            opt.zero_grad()
            out = model(input_ids=ids.to(device), attention_mask=am.to(device)).logits
            loss = lossf(out, lb.to(device)); loss.backward(); opt.step()
    model.eval(); probs = []
    dte = make_loader([texts[i] for i in te_idx], y[te_idx], False)
    with torch.no_grad():
        for ids, am, lb in dte:
            logit = model(input_ids=ids.to(device), attention_mask=am.to(device)).logits
            probs.append(torch.softmax(logit, 1)[:, 1].cpu().numpy())
    return np.concatenate(probs), model

In [ ]:
# 5-fold out-of-fold ViralBERT predictions
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(y))
for k, (tr, te) in enumerate(cv.split(texts, y)):
    p, _ = train_one(tr, te); oof[te] = p
    print(f"fold {k+1}/5 done")
print(f"\nViralBERT  PR-AUC: {average_precision_score(y, oof):.3f}  |  ROC-AUC: {roc_auc_score(y, oof):.3f}")

In [ ]:
# TF-IDF reference on the SAME 5-fold scheme
pipe = Pipeline([("tf", TfidfVectorizer(max_features=2000, min_df=5, ngram_range=(1, 2),
                                        stop_words="english", sublinear_tf=True)),
                 ("lr", LogisticRegression(max_iter=2000, class_weight="balanced"))])
p_tf = cross_val_predict(pipe, pd.Series(texts), y, cv=cv, method="predict_proba")[:, 1]
print(f"TF-IDF     PR-AUC: {average_precision_score(y, p_tf):.3f}  (reference)")

In [ ]:
# fit final ViralBERT on ALL data + save artifacts for serving
_, final_model = train_one(np.arange(len(y)), np.arange(len(y)))
final_model.save_pretrained("/kaggle/working/viralbert")
tok.save_pretrained("/kaggle/working/viralbert")
pd.DataFrame({"video_id": df["video_id"], "is_viral": y, "content_score": oof}) \
  .to_parquet("/kaggle/working/viralbert_oof_scores.parquet", index=False)
print("Saved -> /kaggle/working/viralbert/ (model+tokenizer) and viralbert_oof_scores.parquet")
print("Download both from the Output tab.")